# 面试题：Feature Store 怎样做 Point-in-Time Join，避免特征穿越？

随机 `merge` 用户表和标签表很容易把未来特征送进训练。本 Notebook 手写 event time/available time 双时间合同、版本化特征、Point-in-Time lookup、TTL/default、late arrival、离线 materialization、在线 store、训练服务一致性和快照发布。

不调用 pandas `merge_asof` 或 Feature Store SDK；核心查找使用排序、二分和显式条件，便于看清泄漏发生在哪里。

In [ ]:
import bisect,copy,hashlib,json,math,warnings
from collections import defaultdict
from dataclasses import dataclass
from types import MappingProxyType
warnings.filterwarnings("ignore",message="The pynvml package is deprecated")
import numpy as np
def canonical73(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"))
def sha73(x): return hashlib.sha256(x).hexdigest()
assert bisect.bisect_right([1,3],2)==1

## 1. 三个时间不能混为一个

标签有 `prediction_time`：线上真正发起预测的时刻。特征记录有 `event_time`（业务事实发生）和 `available_time`（经过计算后可被模型读取）。合法特征必须同时满足 `event_time <= prediction_time` 与 `available_time <= prediction_time`。

仅检查 event time 会使用“当时尚未计算完成”的未来知识；仅检查 available time 又可能接受迟到但描述未来事件的数据。backfill 也必须保留原始 available time 或声明重放语义。

In [ ]:
@dataclass(frozen=True)
class FeatureRecord73:
    entity:str; name:str; event_time:int; available_time:int; value:float; version:int
@dataclass(frozen=True)
class LabelRow73:
    row_id:str; entity:str; prediction_time:int; label:int
records73=[
    FeatureRecord73("u1","spend_7d",10,12,20.,1),FeatureRecord73("u1","spend_7d",20,21,35.,1),FeatureRecord73("u1","spend_7d",25,40,90.,1),
    FeatureRecord73("u2","spend_7d",8,9,5.,1),FeatureRecord73("u2","spend_7d",18,18,8.,1),
    FeatureRecord73("u1","country_score",5,5,.7,1),FeatureRecord73("u2","country_score",5,5,.2,1),
]
labels73=[LabelRow73("r1","u1",19,0),LabelRow73("r2","u1",30,1),LabelRow73("r3","u2",20,0)]
assert all(r.event_time<=r.available_time for r in records73) and len({x.row_id for x in labels73})==3
assert records73[2].event_time<30<records73[2].available_time
try: FeatureRecord73("u", "x", 5,4,1.,1); assert False, "dataclass itself does not validate"
except AssertionError as e: assert "dataclass" in str(e)

## 2. 版本化时间索引

同一 `(entity, feature, event_time)` 可能有修订版本；只保留在 cutoff 前已发布的最高 version。索引按 event time 排序，二分找到不晚于 prediction time 的候选，再从近到远检查 available time。

生产中通常按实体分区并保存 `(event_time, created_timestamp)`；这里的列表索引用于展示语义，不代表大规模存储结构。

In [ ]:
class TemporalFeatureIndex73:
    def __init__(self,records):
        grouped=defaultdict(list)
        for r in records:
            if r.event_time>r.available_time or r.version<1 or not math.isfinite(r.value): raise ValueError("feature_record_contract")
            grouped[(r.entity,r.name)].append(r)
        self.rows={k:tuple(sorted(v,key=lambda r:(r.event_time,r.available_time,r.version))) for k,v in grouped.items()}
        self.times={k:tuple(r.event_time for r in v) for k,v in self.rows.items()}
    def lookup(self,entity,name,prediction_time,ttl=None):
        key=(entity,name); rows=self.rows.get(key,()); pos=bisect.bisect_right(self.times.get(key,()),prediction_time)
        eligible=[r for r in rows[:pos] if r.available_time<=prediction_time and (ttl is None or prediction_time-r.event_time<=ttl)]
        if not eligible: return None
        latest_time=max(r.event_time for r in eligible); same=[r for r in eligible if r.event_time==latest_time]; return max(same,key=lambda r:(r.version,r.available_time))
index73=TemporalFeatureIndex73(records73)
assert index73.lookup("u1","spend_7d",19).value==20.
assert index73.lookup("u1","spend_7d",30).value==35. and index73.lookup("u1","spend_7d",30).event_time==20
assert index73.lookup("u1","spend_7d",11) is None and index73.lookup("missing","spend_7d",30) is None
assert index73.lookup("u1","spend_7d",30,ttl=5) is None
try: TemporalFeatureIndex73([FeatureRecord73("u","x",5,4,1.,1)]); raise AssertionError("future availability accepted")
except ValueError as e: assert str(e)=="feature_record_contract"

## 3. 用反例证明 naive latest join 会泄漏

naive join 直接取实体最新记录，`r1@19` 会看到 event=25/available=40 的 90；只按 event time join，`r2@30` 也会看到该记录，虽然线上要到 40 才可用。PIT join 两个条件都拒绝它。

一个好单测必须构造“event 已发生但尚未 available”的记录，否则 available-time bug 永远不会被触发。

In [ ]:
def naive_latest73(entity,name):
    rows=[r for r in records73 if r.entity==entity and r.name==name]
    return max(rows,key=lambda r:(r.event_time,r.version)) if rows else None
def event_only73(entity,name,t):
    rows=[r for r in records73 if r.entity==entity and r.name==name and r.event_time<=t]
    return max(rows,key=lambda r:(r.event_time,r.version)) if rows else None
assert naive_latest73("u1","spend_7d").value==90.
assert event_only73("u1","spend_7d",30).value==90.
assert index73.lookup("u1","spend_7d",30).value==35.
assert event_only73("u1","spend_7d",19).value==20. and index73.lookup("u1","spend_7d",19).value==20.

## 4. 离线训练集 materialization

对每个 label row 和 feature spec 做 PIT lookup，输出值、missing flag、source event/available time。默认值不能抹掉 missingness；模型需要知道 0 是真实值还是没有历史。

materialization 结果要绑定 label snapshot、feature snapshot、spec/TTL 和代码版本，才能重现实验。

In [ ]:
SPECS73={"spend_7d":{"ttl":15,"default":0.},"country_score":{"ttl":100,"default":.5}}
def materialize73(labels,index,specs):
    output=[]
    for row in labels:
        values={}; provenance={}
        for name,spec in specs.items():
            rec=index.lookup(row.entity,name,row.prediction_time,spec["ttl"]); values[name]=spec["default"] if rec is None else rec.value; values[name+"__missing"]=float(rec is None)
            provenance[name]=None if rec is None else (rec.event_time,rec.available_time,rec.version)
        output.append({"row_id":row.row_id,"entity":row.entity,"prediction_time":row.prediction_time,"label":row.label,"values":values,"provenance":provenance})
    return output
training_rows73=materialize73(labels73,index73,SPECS73)
assert training_rows73[0]["values"]["spend_7d"]==20. and training_rows73[1]["values"]["spend_7d"]==35.
assert training_rows73[2]["values"]["spend_7d"]==8. and all(r["values"]["country_score__missing"]==0 for r in training_rows73)
assert all(p is None or p[1]<=row["prediction_time"] for row in training_rows73 for p in row["provenance"].values())
assert len({r["row_id"] for r in training_rows73})==len(labels73)

## 5. Late arrival、修订与 backfill

一个 event=16 的特征到 time=28 才到达：预测 20 不能使用，预测 30 可以使用，但它是否覆盖 event=20 的较新事实取决于“按 event time 取最新”，因此不会覆盖。修订同 event_time 时选择 cutoff 前最高 version。

backfill 若把 available time 改成回放时刻，会得到保守但不同的数据；若伪装成 event time，则制造泄漏。策略必须显式。

In [ ]:
revisions73=records73+[
    FeatureRecord73("u1","spend_7d",16,28,28.,1),
    FeatureRecord73("u1","spend_7d",20,29,36.,2),
    FeatureRecord73("u1","spend_7d",20,35,40.,3),
]
revised_index73=TemporalFeatureIndex73(revisions73)
assert revised_index73.lookup("u1","spend_7d",27).value==35.
assert revised_index73.lookup("u1","spend_7d",30).value==36. and revised_index73.lookup("u1","spend_7d",30).version==2
assert revised_index73.lookup("u1","spend_7d",36).value==40. and revised_index73.lookup("u1","spend_7d",36).version==3
assert revised_index73.lookup("u1","spend_7d",20).event_time==10

## 6. 在线 store 与训练/服务一致性

在线 store 在当前 wall-clock 只发布已 available 的最新 event-time/value/version。请求读取时也要带 prediction time；若直接读“现在最新”，重放历史请求会不同。这里构建给定 as-of time 的在线视图，再与离线 PIT 输出对比。

真实系统要处理 read-after-write、区域复制、TTL、schema 演进和默认值降级，并记录实际命中特征时间。

In [ ]:
class OnlineView73:
    def __init__(self,index,as_of): self.index=index; self.as_of=int(as_of)
    def get(self,entity,name,ttl=None): return self.index.lookup(entity,name,self.as_of,ttl)
online19_73=OnlineView73(index73,19); online30_73=OnlineView73(index73,30)
assert online19_73.get("u1","spend_7d",15).value==training_rows73[0]["values"]["spend_7d"]
assert online30_73.get("u1","spend_7d",15).value==training_rows73[1]["values"]["spend_7d"]
assert OnlineView73(index73,11).get("u1","spend_7d") is None
assert online30_73.get("u1","country_score",100).value==.7

## 7. 自动泄漏审计与质量指标

每行 provenance 应满足两个时间不变量，并统计 missing rate、age、default rate 和 feature freshness 分位数。任何 source available time 晚于 prediction time 都是阻塞错误，而不是普通告警。

数据切分还要按实体/时间策略决定：若目标是未来预测，validation/test 必须在时间上晚于 train。

In [ ]:
def audit73(rows):
    violations=[]; ages=[]; missing=0; total=0
    for row in rows:
        for name,p in row["provenance"].items():
            total+=1
            if p is None: missing+=1; continue
            event,available,version=p; ages.append(row["prediction_time"]-event)
            if event>row["prediction_time"] or available>row["prediction_time"] or version<1: violations.append((row["row_id"],name,p))
    return {"violations":violations,"missing_rate":missing/total,"max_age":max(ages) if ages else None}
audit_result73=audit73(training_rows73)
assert audit_result73["violations"]==[] and 0<=audit_result73["missing_rate"]<=1
leaked_rows73=copy.deepcopy(training_rows73); leaked_rows73[0]["provenance"]["spend_7d"]=(25,40,1)
assert audit73(leaked_rows73)["violations"]==[("r1","spend_7d",(25,40,1))]
assert audit_result73["max_age"]>=0

## 8. 发布快照与面试总结

manifest 绑定 feature specs、event/available/version 语义、完整 record snapshot、label snapshot、default/TTL 和 join 代码版本。服务与训练都从同一 spec 生成读取逻辑，避免手写两份。

面试回答顺序：定义 prediction/event/available time → PIT lookup → late/backfill → offline/online parity → provenance/审计 → 存储与 SLA。核心不是“有个 Feature Store”，而是历史可用性可证明。

In [ ]:
def record_digest73(rows): return sha73(canonical73([r.__dict__ for r in sorted(rows,key=lambda x:(x.entity,x.name,x.event_time,x.available_time,x.version))]).encode())
manifest73={"artifact_id":"pit-features-v1","time_contract":{"event":"fact_time","available":"first_readable_time","cutoff":"prediction_time","conditions":["event<=cutoff","available<=cutoff"]},"specs":SPECS73,"records":record_digest73(records73),"labels":sha73(canonical73([r.__dict__ for r in labels73]).encode()),"join":"latest_event_then_highest_available_version-v1"}
TRUST73=MappingProxyType({manifest73["artifact_id"]:sha73(canonical73(manifest73).encode())})
def load_feature_view73(m,records):
    actual=copy.deepcopy(m); actual["records"]=record_digest73(records)
    if TRUST73.get(actual.get("artifact_id"))!=sha73(canonical73(actual).encode()): raise RuntimeError("untrusted_feature_snapshot")
    return TemporalFeatureIndex73(records)
loaded_index73=load_feature_view73(manifest73,records73)
assert loaded_index73.lookup("u1","spend_7d",30).value==35. and isinstance(TRUST73,MappingProxyType)
forged_records73=records73+[FeatureRecord73("u1","spend_7d",29,29,999.,1)]
try: load_feature_view73(manifest73,forged_records73); raise AssertionError("forged features accepted")
except RuntimeError as e: assert str(e)=="untrusted_feature_snapshot"
print({"rows":len(training_rows73),"audit":audit_result73})

## 9. 复杂度、失败模式与来源

分组排序后单 lookup 二分定位近似 `O(log n)`，但检查 available/version 可能回扫；生产可建复合索引。常见错误：只用 event time、全表 latest join、backfill 改写可用时间、默认值无 missing flag、实体跨 split、在线直接读现在值和特征升级不失效缓存。

- Uber, [Michelangelo Palette: Point-in-time correct joins](https://www.uber.com/blog/michelangelo-palette-feature-engineering/)。
- Feast, [Point-in-time joins](https://docs.feast.dev/getting-started/concepts/point-in-time-joins)。
- Google Cloud, [Feature management and training-serving skew](https://cloud.google.com/architecture/mlops-continuous-delivery-and-automation-pipelines-in-machine-learning)。